# AC-MOT v10 — RUN 1: Baseline_Default
**Account:** any  
**System:** Baseline_Default (standard ByteTrack, fixed conf=0.25)  
**Sequences:** ALL 17  
**Est. time:** ~55 min on T4  
**Saves to:** `/content/drive/MyDrive/VisDrone_Results/`

In [ ]:
!pip install ultralytics motmetrics opencv-python-headless pandas numpy tqdm scipy lap pyyaml -q

import os, time, shutil, gc, yaml
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict, deque
from dataclasses import dataclass

import cv2, numpy as np, pandas as pd, torch, motmetrics as mm
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive

try: torch.backends.cudnn.benchmark = True
except: pass

drive.mount('/content/drive', force_remount=False)

DATASET_ROOT  = Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
SEQ_DIR       = DATASET_ROOT / 'sequences'
ANNOT_DIR     = DATASET_ROOT / 'annotations'
DRIVE_RESULTS = Path('/content/drive/MyDrive/VisDrone_Results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
LOCAL_TMP     = Path('/content/_acmot_tmp')

assert SEQ_DIR.exists(), f'Dataset not found: {SEQ_DIR}'
all_sequences = sorted([d for d in SEQ_DIR.iterdir() if d.is_dir()])
VAL_SEQS = all_sequences  # ALL 17

MODEL_NAME = 'yolov8n.pt'
DEVICE     = '0' if torch.cuda.is_available() else 'cpu'
HALF       = DEVICE != 'cpu'

print(f'Device={DEVICE} | FP16={HALF} | Sequences={len(VAL_SEQS)}')
print('SYSTEM: Baseline_Default')

In [ ]:
# ── Helper functions ─────────────────────────────────────────────

def load_gt(path):
    if not path.exists(): return pd.DataFrame()
    cols = ['frame','id','x','y','w','h','score','cat','trunc','occ']
    df = pd.read_csv(path, header=None, names=cols)
    df = df[df['cat'].isin([1,4,5,6,9])]
    df = df[(df['occ'] < 2) & (df['trunc'] < 2) & (df['score'] == 1)]
    return df.reset_index(drop=True)

def iou_dist(pred, gt):
    if not len(pred) or not len(gt): return np.empty((len(gt), len(pred)))
    ix1 = np.maximum(pred[:,0:1].T, gt[:,0:1]); iy1 = np.maximum(pred[:,1:2].T, gt[:,1:2])
    ix2 = np.minimum(pred[:,2:3].T, gt[:,2:3]); iy2 = np.minimum(pred[:,3:4].T, gt[:,3:4])
    inter = np.maximum(0,ix2-ix1)*np.maximum(0,iy2-iy1)
    ap = (pred[:,2]-pred[:,0])*(pred[:,3]-pred[:,1])
    ag = (gt[:,2]-gt[:,0])*(gt[:,3]-gt[:,1])
    union = ap[np.newaxis,:] + ag[:,np.newaxis] - inter
    return 1.0 - np.where(union>0, inter/union, 0.0)

def hota_approx(tp,fp,fn,ids):
    det_a = tp/max(tp+fp+fn,1)
    ass_a = max(0.0, 1.0 - ids/max(tp,1))
    return float(np.sqrt(det_a*ass_a))

def eval_acc(acc, name='seq'):
    mh = mm.metrics.create()
    s = mh.compute(acc, metrics=['mota','idf1','num_switches','recall','precision',
                                  'num_misses','num_false_positives','num_matches'], name=name)
    r = s.iloc[0]
    return dict(mota=float(r['mota']), idf1=float(r['idf1']), recall=float(r['recall']),
                precision=float(r['precision']), ids=int(r['num_switches']),
                fn=int(r['num_misses']), fp=int(r['num_false_positives']),
                matches=int(r['num_matches']),
                hota=hota_approx(int(r['num_matches']),int(r['num_false_positives']),
                                 int(r['num_misses']),int(r['num_switches'])))

def reset_tracker(model):
    if getattr(model,'predictor',None) is not None: model.predictor = None

print('Helpers ready')

In [ ]:
# ── RUN ──────────────────────────────────────────────────────────
ts      = datetime.now().strftime('%Y%m%d_%H%M%S')
run_tag = f'acmot_v10_BASELINE_DEFAULT_17seq_{ts}'

model = YOLO(MODEL_NAME)
if HALF: model.model.half()

rows = []
for seq in tqdm(VAL_SEQS, desc='Baseline_Default'):
    gt = load_gt(ANNOT_DIR / f'{seq.name}.txt')
    if gt.empty or not list(seq.glob('*.jpg')): continue

    LOCAL_TMP.mkdir(exist_ok=True)
    local_seq = LOCAL_TMP / seq.name
    if local_seq.exists(): shutil.rmtree(local_seq)
    shutil.copytree(seq, local_seq)
    frames = sorted(local_seq.glob('*.jpg'))

    reset_tracker(model)
    acc = mm.MOTAccumulator(auto_id=True)
    times = []

    for idx, fp in enumerate(frames, start=1):
        t0  = time.perf_counter()
        img = cv2.imread(str(fp))
        if img is None: continue

        res = model.track(source=img, tracker='bytetrack.yaml',
                          conf=0.25, iou=0.45, imgsz=640,
                          half=HALF, persist=True, verbose=False, device=DEVICE)
        times.append(time.perf_counter()-t0)

        pred_ids   = res[0].boxes.id.cpu().numpy().astype(int) if res[0].boxes.id is not None else np.array([],dtype=int)
        pred_boxes = res[0].boxes.xyxy.cpu().numpy()           if res[0].boxes.id is not None else np.empty((0,4))

        gt_f = gt[gt['frame']==idx]
        gt_ids = gt_f['id'].values
        gt_boxes = (np.column_stack([gt_f['x'].values, gt_f['y'].values,
                                     gt_f['x'].values+gt_f['w'].values,
                                     gt_f['y'].values+gt_f['h'].values])
                    if len(gt_f) else np.empty((0,4)))
        dist = iou_dist(pred_boxes, gt_boxes)
        acc.update(gt_ids, pred_ids, dist if dist.size else np.empty((len(gt_ids),len(pred_ids))))

    shutil.rmtree(local_seq, ignore_errors=True)
    m   = eval_acc(acc, seq.name)
    fps = 1.0/np.mean(times) if times else 0.0
    rows.append(dict(run_tag=run_tag, system='Baseline_Default', sequence=seq.name,
                     frames=len(frames), fps=round(fps,2), **m))
    tqdm.write(f"Baseline_Default {seq.name[:28]:28s} MOTA={m['mota']:.3f} IDF1={m['idf1']:.3f} IDS={m['ids']:4d} FPS={fps:.1f}")

if HALF: torch.cuda.empty_cache()
gc.collect()

df = pd.DataFrame(rows)
out_path = DRIVE_RESULTS / f'{run_tag}_per_sequence.csv'
df.to_csv(out_path, index=False)
print(f'\nSaved -> {out_path}')
print(f'\nMean MOTA={df["mota"].mean():.4f}  IDF1={df["idf1"].mean():.4f}  IDS={df["ids"].sum()}  FPS={df["fps"].mean():.1f}')